# ✂️ CutKit GPU backend (free Colab T4)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lyraofweb3/cutkit/blob/main/colab/CutKit_GPU.ipynb)

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. Copy the printed **URL** and **token** into CutKit → ⚙️ Settings → *Colab GPU*.

Keep this tab open: Colab stops the server when the session ends. Models download on first use.

| Endpoint | Model | Licence |
|---|---|---|
| `/video/remove-bg`, `/remove-bg` | BiRefNet (ZhengPeng7/BiRefNet) + ffmpeg | MIT |
| `/upscale` | Real-ESRGAN General x4v3 (ONNX) | BSD-3-Clause |
| `/generate` | SDXL base 1.0 + LCM-LoRA (4 steps) | CreativeML OpenRAIL++-M (commercial use allowed, use restrictions apply) |

In [ ]:
#@title 1 · Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ No GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
#@title 2 · Install dependencies (~2 min)
!pip -q install fastapi "uvicorn[standard]" python-multipart onnxruntime-gpu timm kornia einops diffusers peft accelerate "transformers<5"
!apt-get -qq install -y ffmpeg > /dev/null
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!wget -q -O /content/server.py https://raw.githubusercontent.com/lyraofweb3/cutkit/main/colab/server.py
print("✅ installed")

In [ ]:
#@title 3 · Start server + public tunnel
import os, secrets, subprocess, threading, time, re, sys
os.environ["CUTKIT_TOKEN"] = os.environ.get("CUTKIT_TOKEN") or secrets.token_urlsafe(24)
os.environ.setdefault("CUTKIT_ORIGINS", "https://cutkit-production-c14f.up.railway.app,https://lyraofweb3.github.io,http://localhost:5173")
sys.path.insert(0, "/content")
import uvicorn, server

threading.Thread(target=lambda: uvicorn.run(server.app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(3)
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL, deadline = None, time.time() + 90
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m and not URL: URL = m.group(0)
    if URL and "Registered tunnel connection" in line: break
    if time.time() > deadline: print("⚠️ tunnel slow to register; URL may take a minute"); break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print("\n================ CutKit GPU ready ================")
print("URL:  ", URL)
print("Token:", os.environ["CUTKIT_TOKEN"])
print("Paste both into CutKit → ⚙️ Settings → Colab GPU")
print("==================================================")

In [ ]:
#@title 4 · (Optional) Pre-load models now so the first request is fast
preload_video = True #@param {type:"boolean"}
preload_upscale = True #@param {type:"boolean"}
preload_generate = False #@param {type:"boolean"}
if preload_video: server.birefnet(); print("BiRefNet ready")
if preload_upscale: server.esrgan(); print("Real-ESRGAN ready")
if preload_generate: server.generator(); print("SDXL+LCM ready")

In [ ]:
#@title 5 · Keep alive (leave running)
import requests
while True:
    try: print(time.strftime("%H:%M:%S"), requests.get("http://127.0.0.1:8000/health", timeout=5).json())
    except Exception as e: print("health error", e)
    time.sleep(300)